## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-20-04-51-00 +0000,wsj,"TPG, Corpay to Buy Finance Platform Bill for A...",https://www.wsj.com/business/deals/tpg-corpay-...
1,2026-11-20-04-43-04 +0000,startribune,"Kirill Kaprizov, Quinn Hughes extend points st...",https://www.startribune.com/minnesota-wild-cal...
2,2026-11-20-04-35-51 +0000,bbc,"UK and EU to agree youth mobility scheme, BBC ...",https://www.bbc.co.uk/news/articles/c623j4kpg9...
3,2026-11-20-04-33-09 +0000,nypost,USDA impacted by ‘cybersecurity incident’ resu...,https://nypost.com/2026/11/19/us-news/usda-imp...
4,2026-11-20-04-15-35 +0000,nypost,2-year-old lost in the desert for 7 hours wear...,https://nypost.com/2026/11/19/us-news/2-year-o...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2641/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
156,trump,56
44,anthropic,40
326,putin,28
325,xi,19
48,ai,19


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
234,2026-11-19-19-03-56 +0000,nypost,China sent its tallest official to tower over ...,https://nypost.com/2026/11/19/world-news/china...,151
246,2026-11-19-18-02-00 +0000,wsj,"Trump, Xi, Putin End Historic Summit With Denu...",https://www.wsj.com/world/trump-xi-and-putin-c...,147
411,2026-11-19-10-25-01 +0000,cbc,"Xi, Putin and Trump walked into a room. But it...",https://www.cbc.ca/news/world/xi-putin-trump-9...,144
67,2026-11-20-00-51-01 +0000,nyt,"Trump, Xi and Putin Seek to Project Power at T...",https://www.nytimes.com/live/2026/11/18/world/...,144
103,2026-11-19-23-23-22 +0000,wapo,"In summit with Trump and Putin, Xi put himself...",https://www.washingtonpost.com/world/2026/11/1...,139


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
234,151,2026-11-19-19-03-56 +0000,nypost,China sent its tallest official to tower over ...,https://nypost.com/2026/11/19/world-news/china...
89,106,2026-11-20-00-00-00 +0000,wsj,Anthropic’s Mega IPO Lives Up to Wall Street H...,https://www.wsj.com/finance/stocks/anthropic-i...
204,94,2026-11-19-20-07-02 +0000,bbc,Myanmar committed genocide against Rohingya mi...,https://www.bbc.co.uk/news/articles/c804wzp20d...
25,85,2026-11-20-02-54-11 +0000,nypost,White House taps Kirsten Davies as new cyber c...,https://nypost.com/2026/11/19/us-news/wh-names...
144,72,2026-11-19-22-06-00 +0000,wsj,Senate Republicans are suspending their effort...,https://www.wsj.com/politics/policy/pentagon-m...
8,58,2026-11-20-04-12-00 +0000,wsj,The Senate passed a bipartisan bill designed t...,https://www.wsj.com/politics/policy/senate-pas...
399,56,2026-11-19-11-00-00 +0000,latimes,"High tides, big storms: Next week will be key ...",https://www.latimes.com/california/story/2026-...
99,45,2026-11-19-23-33-51 +0000,bbc,Anthropic becomes one of world's most valuable...,https://www.bbc.co.uk/news/articles/c3x29ggenz...
97,42,2026-11-19-23-41-09 +0000,nypost,California AG Rob Bonta serves Nvidia with sub...,https://nypost.com/2026/11/19/business/califor...
119,42,2026-11-19-22-48-22 +0000,bbc,Judge finds Thomas Markle mentally fit to stan...,https://www.bbc.co.uk/news/articles/crgj35ldz2...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
